# Data Cleaning & Structural Validation – Customer Churn Dataset

**Objective:** Transform the raw customer churn dataset into a reliable format for analysis using Python and Pandas.

### Cleaning tasks
- Inspect missing values
- Identify duplicate records
- Validate data types
- Standardize column headers
- Clean categorical strings
- Validate numeric ranges
- Handle missing values with documented reasoning
- Export the cleaned CSV

**Note:** The dataset does not contain a date column, so date-format standardization is **not applicable** to this dataset.

In [ ]:
import pandas as pd

input_file = "customer_churn_sample (1)(1).csv"
df = pd.read_csv(input_file)

print("Dataset shape:", df.shape)
display(df.head())

## 1. Initial inspection

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

### Initial inspection result

The dataset contains **15 rows and 11 columns**. No missing values and no exact duplicate rows were found. The numeric fields are already stored as numeric data types, while ID and categorical fields are stored as text.

Because there are no missing values, no imputation or row deletion is required. This avoids changing valid observed data unnecessarily.

## 2. Check duplicate customer IDs

In [ ]:
print("Duplicate CustomerID values:", df["CustomerID"].duplicated().sum())
print(df["CustomerID"].nunique(), "unique CustomerIDs out of", len(df), "rows")

## 3. Standardize column headers

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.replace(r'(?<=[a-z0-9])(?=[A-Z])', '_', regex=True)
    .str.replace(r'[^A-Za-z0-9]+', '_', regex=True)
    .str.strip('_')
    .str.lower()
)

print("Standardized columns:")
print(df.columns.tolist())

The headers are converted to lowercase `snake_case`, which is consistent and convenient for Python analysis.

## 4. Clean categorical/string columns

In [ ]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()

categorical_columns = [
    "gender", "subscription_type", "contract_type",
    "payment_method", "churn"
]

for col in categorical_columns:
    print(f"{col}: {df[col].unique().tolist()}")

No inconsistent categorical values were found after trimming whitespace. The observed categories are kept as they represent the dataset's original values.

## 5. Validate and convert numeric data types

In [ ]:
numeric_columns = [
    "age", "tenure_months", "monthly_charges",
    "total_charges", "support_tickets"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_columns].dtypes)

Using `errors='coerce'` makes invalid numeric values visible as missing values. The dataset is then checked again for missing values.

In [ ]:
print("Missing values after type conversion:")
print(df.isnull().sum())

## 6. Handle missing values and duplicates

In [ ]:
# Remove exact duplicate rows if any exist.
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
removed_duplicates = before - len(df)

print("Duplicate rows removed:", removed_duplicates)
print("Remaining missing values:", int(df.isnull().sum().sum()))

### Missing-value reasoning

The original dataset contains **0 missing values** in every column. Therefore:

- No numeric values were imputed.
- No categorical values were imputed.
- No rows were dropped because of null values.

This is documented rather than artificially filling valid data.

## 7. Structural validation

In [ ]:
# Range checks
assert df["age"].between(0, 120).all()
assert (df["tenure_months"] >= 0).all()
assert (df["monthly_charges"] >= 0).all()
assert (df["total_charges"] >= 0).all()
assert (df["support_tickets"] >= 0).all()

# Categorical checks
assert set(df["gender"].dropna().unique()) <= {"Female", "Male"}
assert set(df["subscription_type"].dropna().unique()) <= {"Basic", "Pro", "Enterprise"}
assert set(df["contract_type"].dropna().unique()) <= {"Month-to-Month", "One Year", "Two Year"}
assert set(df["payment_method"].dropna().unique()) <= {"Credit Card", "Bank Transfer", "UPI", "Debit Card"}
assert set(df["churn"].dropna().unique()) <= {"Yes", "No"}

print("All structural validation checks passed.")

## 8. Final validation report

In [ ]:
print("Final shape:", df.shape)
print("\nFinal data types:")
print(df.dtypes)
print("\nFinal missing values:", int(df.isnull().sum().sum()))
print("Final duplicate rows:", int(df.duplicated().sum()))
display(df.head(10))

## 9. Export cleaned dataset

The cleaned dataset is exported as `customer_churn_cleaned.csv` for submission.

In [ ]:
output_file = "customer_churn_cleaned.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")

## Final summary

- **Rows:** 15
- **Columns:** 11
- **Missing values:** 0
- **Exact duplicate rows:** 0
- **Duplicate customer IDs:** 0
- **Invalid numeric values:** 0
- **Date column:** Not present, so date-format standardization is not applicable
- **Column headers:** Standardized to lowercase snake_case
- **Categorical strings:** Trimmed and validated
- **Cleaned CSV:** `customer_churn_cleaned.csv`

The cleaned dataset is ready for analysis.